# RAG with Chunking Strategies

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
#from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModel
device = torch.device("cuda" if ... else "cpu")

In [2]:
import nltk

In [3]:
# if it doesnt work
# !pip install nltk

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Load Model

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = .......(..., cache_dir="/tmp")
model = .......(..., cache_dir="/tmp").to(device)

## Load and Preprocess Text

In [5]:
script_path = "shrek.txt"

with open(script_path, 'r', encoding='utf-8') as file:
    script_text = file.read()
import re
script_text = re.sub(' +', ' ', script_text)

## Chunking Strategies

In [6]:
def create_fixed_size_chunks(text, chunk_size=1000, overlap=0):
    chunks = []
    start = 0
    ...
    return chunks

In [7]:
from nltk.tokenize import sent_tokenize
#nltk.data.find('tokenizers/punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/mwm/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
def create_sentence_chunks(text, sentences_per_chunk=10):
    sentences = ....(text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i+sentences_per_chunk])
        chunks.append(chunk)
    return chunks

## Embeddings

In [9]:
chunks_sentences = create_sentence_chunks(script_text, sentences_per_chunk=15)

In [10]:
chunks_fixed_size = create_fixed_size_chunks(script_text)
chunks_fixed_size_overlapping = create_fixed_size_chunks(script_text, overlap=50)

In [ ]:
def get_embeddings(texts, tokenizer, model):
    encoded_inputs = ...(..., padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = ...(...)
    token_embeddings = outputs.last_hidden_state
    embeddings = torch....(token_embeddings, dim=1)
    return embeddings.cpu().numpy()

In [12]:
embeddings_fixed_size = get_embeddings(chunks_fixed_size, tokenizer, model)
embeddings_fixed_size_overlapping = get_embeddings(chunks_fixed_size_overlapping, tokenizer, model)
embeddings_sentences= get_embeddings(chunks_sentences, tokenizer, model)

## Retrieval

In [13]:
cosine_similarity = ...
dot_product_similarity = ...
euclidean_similarity = ...

In [ ]:
def retrieve_chunks(query_embedding, chunk_embeddings, chunks, top_k=3, similarity_fn=cosine_similarity):
    similarities = []
    for i, chunk_embedding in enumerate(chunk_embeddings):
        similarity = ...(query_embedding, chunk_embedding)
        similarities.append(...)
    similarities.sort(...)
    return similarities[:top_k]

In [15]:
sample_query = "Who is farquads wife."+" DONKEY"

query_embedding = get_embeddings([sample_query], tokenizer, model)[0]
em = embeddings_fixed_size_overlapping
ch = chunks_fixed_size_overlapping
retrieve_chunks(query_embedding, em, ch, top_k=3, similarity_fn=cosine_similarity)

[(62,
  np.float32(0.4198594),
  "shock. He looks past her and \n spots a group approaching.) Ah, right \n on time. Princess, I've brought you \n a little something.\n \n Farquaad has arrived with a group of his men. He looks very regal \n sitting up on his horse. You would never guess that he's only \n like 3 feet tall. Donkey wakes up with a yawn as the soldiers \n march by.\n \n DONKEY\n What'd I miss? What'd I miss? (spots \n the soldiers) (muffled) Who said that? \n Couldn't have been the donkey.\n \n FARQUAAD\n Princess Fiona.\n\n SHREK\n As promised. Now hand it over.\n\n FARQUAAD\n Very well, ogre. (holds out a piece \n of paper) The deed to your swamp, cleared \n out, as agreed. Take it and go before \n I change my mind. (Shrek takes the paper) \n Forgive me, Princess, for startling \n you, but you startled me, for I have \n never seen such a radiant beauty before. \n I'm Lord Farquaad.\n \n FIONA\n Lord Farquaad? Oh, no, no. (Farquaad \n snaps his fingers) Forgive me, my lord